<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs all required libraries for running the notebook.

In [ ]:
!pip install numpy scipy vectorbt

vectorbt depends on yfinance for market data downloads, and it will pull in pandas and NumPy automatically. If you run into installation issues with vectorbt (especially on Apple Silicon or certain Linux distributions), check the vectorbt docs for conda-based installation instructions.

## Imports and setup

NumPy handles array operations and parameter ranges. scipy.stats provides the statistical test we use at the end to compare in-sample and out-of-sample performance. vectorbt is the backtesting engine that runs over a million simulations in seconds by vectorizing portfolio logic with pandas and NumPy under the hood.

In [ ]:
import numpy as np
import scipy.stats as stats
import vectorbt as vbt

## Download data and split into rolling windows

This creates an array of candidate moving average window lengths from 10 to 49 days. vectorbt will test every possible pair of these windows as fast and slow moving average parameters.

In [ ]:
windows = np.arange(10, 50)

We download Apple's daily closing prices from Yahoo Finance using vectorbt's built-in data interface.

In [ ]:
price = vbt.YFData.download('AAPL').get('Close')

This is the core of walk-forward optimization. We split the price series into 30 overlapping windows, each two years long, where the last 180 days of each window are reserved as out-of-sample data and the rest is in-sample.

In [ ]:
(in_price, in_indexes), (out_price, out_indexes) = (
    price.vbt.rolling_split(
        n=30,
        window_len=365 * 2,
        set_lens=(180,),
        left_to_right=False,
    )
)

The rolling split is what separates walk-forward optimization from a naive backtest. Instead of one train/test split, we get 30 overlapping windows that slide across time. Each window optimizes parameters on in-sample data, then validates on out-of-sample data the strategy has never seen. This mirrors how professionals stress-test strategies across different market regimes rather than relying on a single lucky period.

## Define simulation and parameter selection functions

This function tests every combination of fast and slow moving average windows on a given price series. It generates crossover signals (buy when the fast MA crosses above the slow MA, sell when it crosses below) and returns the Sharpe ratio for each parameter combination across all splits.

In [ ]:
def simulate_all_params(price, windows, **kwargs):
    fast_ma, slow_ma = vbt.MA.run_combs(
        price, windows, r=2, short_names=["fast", "slow"]
    )
    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)

    pf = vbt.Portfolio.from_signals(
        price, entries, exits, **kwargs
    )
    return pf.sharpe_ratio()

The run_combs method generates all pairwise combinations of window lengths and runs them simultaneously as vectorized array operations. This is why vectorbt can evaluate hundreds of thousands of parameter combinations in seconds instead of looping through them one at a time. The Sharpe ratio measures risk-adjusted return, which is a better optimization target than raw profit because it penalizes strategies that only win by taking on excessive volatility.

This function finds the best-performing parameter combination within each of the 30 rolling splits. It groups results by split index and picks the combination with the highest (or lowest) metric value.

In [ ]:
def get_best_index(performance, higher_better=True):
    if higher_better:
        return performance[
            performance.groupby('split_idx').idxmax()
        ].index
    return performance[
        performance.groupby('split_idx').idxmin()
    ].index

This helper extracts the actual window values (fast or slow) from the multi-level index that vectorbt uses to track parameter combinations across splits.

In [ ]:
def get_best_params(best_index, level_name):
    return best_index.get_level_values(level_name).to_numpy()

This function runs a single simulation per split using one specific pair of fast and slow windows (rather than all combinations). We use it to test the best in-sample parameters on out-of-sample data.

In [ ]:
def simulate_best_params(
    price, best_fast_windows, best_slow_windows, **kwargs
):
    fast_ma = vbt.MA.run(
        price, window=best_fast_windows, per_column=True
    )
    slow_ma = vbt.MA.run(
        price, window=best_slow_windows, per_column=True
    )

    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)

    pf = vbt.Portfolio.from_signals(
        price, entries, exits, **kwargs
    )
    return pf.sharpe_ratio()

The per_column=True argument is critical here. It tells vectorbt to apply a different window length to each split column rather than broadcasting one window across all columns. Each split gets the specific best parameters we found during in-sample optimization, so we are genuinely testing whether those parameters generalize to unseen data.

## Run walk-forward optimization and test results

We run all parameter combinations on the in-sample data across all 30 splits. The direction="both" argument allows both long and short positions, and freq="d" tells vectorbt the data is daily.

In [ ]:
in_sharpe = simulate_all_params(
    in_price,
    windows,
    direction="both",
    freq="d",
)

We extract the best-performing parameter pair from each of the 30 in-sample windows and collect them into arrays of fast and slow window lengths.

In [ ]:
in_best_index = get_best_index(in_sharpe)

In [ ]:
in_best_fast_windows = get_best_params(
    in_best_index,
    'fast_window',
)
in_best_slow_windows = get_best_params(
    in_best_index,
    'slow_window',
)
in_best_window_pairs = np.array(
    list(
        zip(
            in_best_fast_windows,
            in_best_slow_windows,
        )
    )
)

Now we take those best in-sample parameters and run them on the out-of-sample data. This is the moment of truth: if the strategy truly found a real pattern, these Sharpe ratios should hold up reasonably well compared to the in-sample results.

In [ ]:
out_test_sharpe = simulate_best_params(
    out_price,
    in_best_fast_windows,
    in_best_slow_windows,
    direction="both",
    freq="d",
)

We collect the in-sample best Sharpe ratios and the out-of-sample test Sharpe ratios, then run a one-sided t-test to check whether the out-of-sample performance is statistically comparable to in-sample performance.

In [ ]:
in_sample_best = in_sharpe[in_best_index].values
out_sample_test = out_test_sharpe.values

In [ ]:
t, p = stats.ttest_ind(
    a=out_sample_test,
    b=in_sample_best,
    alternative="greater",
)

The t-test asks a precise question: is the out-of-sample performance significantly greater than or equal to the in-sample performance? A high p-value here is actually expected and not alarming, because in-sample results are almost always inflated by overfitting. What you want to watch for is whether the out-of-sample Sharpe ratios collapse to zero or go negative, which would mean the strategy memorized noise rather than finding a real edge. This statistical check replaces gut feeling with a repeatable, quantitative answer before you risk real capital.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.